# Order status analytics: Kafka + PySpark Structured Streaming
Flow: Kafka topic `order-status` -> parse JSON -> window + group -> results table.
Run the cells in order. Cell 7 (Task 4) should be run after the stream has collected a few minutes of data.

In [1]:
# CELL 1: Start Spark with the Kafka connector
# (restart the kernel first if a Spark session already exists)
import pyspark
from pyspark.sql import SparkSession, functions as F

ver = pyspark.__version__                                   # installed Spark version
scala = "2.13" if int(ver.split(".")[0]) >= 4 else "2.12"    # Scala build used by that Spark

spark = (SparkSession.builder.appName("OrderAnalytics")
         .config("spark.jars.packages", f"org.apache.spark:spark-sql-kafka-0-10_{scala}:{ver}")  # Kafka connector
         .config("spark.sql.session.timeZone", "UTC")        # event_time is generated in UTC
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

In [2]:
# CELL 2: Read the stream from Kafka and prepare it for all tasks

# Structure of each JSON message
SCHEMA = ("order_id string, restaurant_id string, delivery_area string, status string, "
          "estimated_delivery_time string, cuisine string, event_time string")

events = (spark.readStream.format("kafka")
          .option("kafka.bootstrap.servers", "localhost:9092")   # Kafka broker
          .option("subscribe", "order-status")                   # topic name
          .option("startingOffsets", "earliest")                 # also read messages sent earlier
          .load()
          # Kafka value is bytes: convert to text, then parse JSON into columns
          .select(F.from_json(F.col("value").cast("string"), SCHEMA).alias("e")).select("e.*")
          # use the time the event happened, not the time it arrived
          .withColumn("event_time", F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss"))
          # accept events up to 2 minutes late; older state is dropped
          .withWatermark("event_time", "2 minutes")
          # remove duplicate messages of the same order status
          .dropDuplicates(["order_id", "status", "event_time"]))

# Helper: run a streaming query and keep its result in an in-memory table named `name`
def run(df, name):
    return df.writeStream.queryName(name).outputMode("update").format("memory").start()

In [3]:
# CELL 3 - TASK 1: Order count by status and delivery area, 1-minute windows
t1 = (events
      .groupBy(F.window("event_time", "1 minute"), "delivery_area", "status")   # 1-min tumbling window
      .count().withColumnRenamed("count", "order_count"))
q1 = run(t1, "task1")

In [4]:
# CELL 4 - TASK 2: Areas with many delayed or cancelled orders (1-minute windows)
# Assumed thresholds (no real data available to learn them from)
MIN_ORDERS = 10      # ignore windows with too few orders (avoids noisy small samples)
MIN_PROBLEM = 15     # alert if at least this many delayed/cancelled orders ...
MIN_RATIO = 0.30     # ... or if at least 30% of orders are delayed/cancelled

is_problem = F.col("status").isin("DELAYED", "CANCELLED")

t2 = (events
      .groupBy(F.window("event_time", "1 minute"), "delivery_area")
      .agg(F.approx_count_distinct("order_id").alias("total_orders"),                              # all orders seen
           F.approx_count_distinct(F.when(is_problem, F.col("order_id"))).alias("problem_orders"))  # delayed/cancelled only
      .withColumn("problem_ratio", F.round(F.col("problem_orders") / F.col("total_orders"), 2))
      .filter((F.col("total_orders") >= MIN_ORDERS) &
              ((F.col("problem_orders") >= MIN_PROBLEM) | (F.col("problem_ratio") >= MIN_RATIO))))
q2 = run(t2, "task2")

In [5]:
# CELL 5 - TASK 3: Order count by cuisine and area, 2-minute windows
# Only PLACED events are counted, so each order is counted once (as new demand)
t3 = (events.filter("status = 'PLACED'")
      .groupBy(F.window("event_time", "2 minutes"), "delivery_area", "cuisine")
      .count().withColumnRenamed("count", "order_count"))
q3 = run(t3, "task3")

In [8]:
# CELL 6: View results of Tasks 1-3 (wait 1-2 minutes after starting; re-run to refresh)
for name in ["task1", "task2", "task3"]:
    print(name)
    spark.table(name).orderBy(F.desc("window")).show(15, truncate=False)   # newest windows first

task1
+------------------------------------------+-------------+---------+-----------+
|window                                    |delivery_area|status   |order_count|
+------------------------------------------+-------------+---------+-----------+
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Shirva       |DELIVERED|1          |
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Brahmavar    |DELIVERED|1          |
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Kundapura    |DELIVERED|1          |
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Udupi        |DELIVERED|2          |
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Manipal      |DELIVERED|2          |
|{2026-10-14 10:30:00, 2026-10-14 10:31:00}|Karkala      |DELIVERED|1          |
|{2026-10-14 10:29:00, 2026-10-14 10:30:00}|Shirva       |DELAYED  |2          |
|{2026-10-14 10:29:00, 2026-10-14 10:30:00}|Manipal      |CANCELLED|1          |
|{2026-10-14 10:29:00, 2026-10-14 10:30:00}|Shirva       |PLACED   |8          |
|{2026-10-14 10:29:00,

In [9]:
# CELL 7 - TASK 4: Flag areas where demand is high (threshold learned from the real data)
# Run this after a few minutes of data has been collected.
# Idea: no fixed number. Threshold = K x the average demand of an area per 2-minute window.
K = 1.5   # an area is 'high demand' if it is 50% above the average area

# Demand per area per window = total new orders from the Task 3 results (all cuisines added up)
demand = spark.sql("SELECT window, delivery_area, SUM(order_count) AS new_orders "
                   "FROM task3 GROUP BY window, delivery_area")

threshold = demand.agg(F.avg("new_orders")).first()[0] * K   # calculated from the data
print("Calculated demand threshold:", round(threshold))

demand.filter(F.col("new_orders") > threshold).orderBy(F.desc("window")).show(truncate=False)

Calculated demand threshold: 89
+------------------------------------------+-------------+----------+
|window                                    |delivery_area|new_orders|
+------------------------------------------+-------------+----------+
|{2026-10-14 10:28:00, 2026-10-14 10:30:00}|Manipal      |97        |
|{2026-10-14 10:26:00, 2026-10-14 10:28:00}|Udupi        |136       |
|{2026-10-14 10:26:00, 2026-10-14 10:28:00}|Manipal      |137       |
|{2026-10-14 10:24:00, 2026-10-14 10:26:00}|Manipal      |135       |
|{2026-10-14 10:24:00, 2026-10-14 10:26:00}|Udupi        |132       |
|{2026-10-14 10:22:00, 2026-10-14 10:24:00}|Udupi        |148       |
|{2026-10-14 10:22:00, 2026-10-14 10:24:00}|Manipal      |131       |
|{2026-10-14 10:20:00, 2026-10-14 10:22:00}|Udupi        |143       |
|{2026-10-14 10:20:00, 2026-10-14 10:22:00}|Manipal      |149       |
|{2026-10-14 10:18:00, 2026-10-14 10:20:00}|Udupi        |107       |
|{2026-10-14 10:18:00, 2026-10-14 10:20:00}|Manipal      |

In [ ]:
# CELL 8: Stop all streaming queries when finished
for q in spark.streams.active:
    q.stop()